In [8]:
import psutil
import time
import threading
import datetime
import json
import xgboost as xgb
import matplotlib.pyplot as plt

import pyspark
import os
import pandas as pd
from torchvision.io import read_image
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import datasets, models
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import pyarrow
from io import BytesIO
import base64



from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import FloatType

import math
from statistics import NormalDist

import sys
sys.path.append('../')

from dbms_benchmark_parent_classes.dbms_parent import DMBSParent

class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = torch.nn.Sequential(
            # Input = 3 x 32 x 32, Output = 32 x 32 x 32
            # new input 3x600x375 output=
            torch.nn.Conv2d(in_channels=3, out_channels=6, kernel_size=3, padding='same'),
            torch.nn.ReLU(),
            # Input = 32 x 32 x 32, Output = 32 x 16 x 16
            torch.nn.MaxPool2d(kernel_size=2),

            # Input = 32 x 16 x 16, Output = 64 x 16 x 16
            torch.nn.Conv2d(in_channels=6, out_channels=12, kernel_size=3, padding='same'),
            torch.nn.ReLU(),
            # Input = 64 x 16 x 16, Output = 64 x 8 x 8
            torch.nn.MaxPool2d(kernel_size=2),

            # Input = 64 x 8 x 8, Output = 64 x 8 x 8
            torch.nn.Conv2d(in_channels=12, out_channels=24, kernel_size=3, padding='same'),
            torch.nn.ReLU(),
            # Input = 64 x 8 x 8, Output = 64 x 4 x 4
            torch.nn.MaxPool2d(kernel_size=2),

            # Input = 64 x 8 x 8, Output = 64 x 8 x 8
            torch.nn.Conv2d(in_channels=24, out_channels=48, kernel_size=3, padding='same'),
            torch.nn.ReLU(),
            # Input = 64 x 8 x 8, Output = 64 x 4 x 4
            torch.nn.MaxPool2d(kernel_size=2),

            # Input = 64 x 8 x 8, Output = 64 x 8 x 8
            torch.nn.Conv2d(in_channels=48, out_channels=96, kernel_size=3, padding='same'),
            torch.nn.ReLU(),
            # Input = 64 x 8 x 8, Output = 64 x 4 x 4
            torch.nn.MaxPool2d(kernel_size=2),

            torch.nn.Flatten(),

            torch.nn.Linear(96 * 11 * 18, 128),
            torch.nn.ReLU(),
            torch.nn.BatchNorm1d(128),

            # torch.nn.Linear(128, 128),
            # torch.nn.ReLU(),
            # torch.nn.BatchNorm1d(128),

            torch.nn.Linear(128, 2),
            torch.nn.ReLU(),
            # torch.nn.BatchNorm1d(2),
            # torch.nn.Dropout(0.2),

            torch.nn.Softmax(dim=1)

        )

    def forward(self, x):
        return self.model(x)



class SparkDB_Naive(DMBSParent):
    def __init__(self, target_table, run_num):
        super().__init__()
        self.target_table = target_table
        self.run_num = run_num
        self.system = "spark"
        self.udf_type = "naive"

        #now = datetime.datetime.now()

    def setup(self):
        self.spark = pyspark.sql.SparkSession.builder.master("local[4]").appName("spark_naive").config(
            "spark.executor.memory", "15g").config("spark.driver.memory", "15g").getOrCreate()

        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model = CNN().to(device)
        model.load_state_dict(torch.load('../mymodel', map_location=torch.device(device)))
        model.eval()  # Set the model to evaluation mode

        bc_model = self.spark.sparkContext.broadcast(model)

        def predict_tensor(base64image):
            img_data = base64.b64decode(base64image)
            image = Image.open(BytesIO(img_data))
            tensor = ToTensor()(image).float()

            with torch.no_grad():  # Disable gradient calculation for inference
                prediction = model(tensor.unsqueeze(0).to(device))
                _, y_pred = torch.max(prediction.data, 1)

            return float(y_pred)

        predict_udf = udf(predict_tensor, FloatType())

        self.spark.udf.register("predict_udf", predict_udf)

        xgboostmodel = xgb.XGBClassifier(enable_categorical=True)

        xgboostmodel.load_model('../xgboost_model.model')

        def predict_xgboost(*features):
            # feature_list = list(features)
            data = [
                {
                    'gender': features[0],
                    'ethnicity': features[1],
                    'class': features[2],
                    'eye_color': features[3],
                    'hair_color': features[4],
                    'is_donor': features[5],
                    'is_veteran': features[6],
                    'height_inches': features[7],
                    'weight_lbs': features[8]
                }
            ]
            df = pd.DataFrame(data)

            df['gender'] = df['gender'].astype('category')
            df['ethnicity'] = df['ethnicity'].astype('category')
            df['class'] = df['class'].astype('category')
            df['eye_color'] = df['eye_color'].astype('category')
            df['hair_color'] = df['hair_color'].astype('category')
            df['is_donor'] = df['is_donor'].astype('category')
            df['is_veteran'] = df['is_veteran'].astype('category')
            df['height_inches'] = df['height_inches'].astype('int64')
            df['weight_lbs'] = df['weight_lbs'].astype('int64')

            # feature_np = np.array(feature_list, dtype=object)

            y_pred = xgboostmodel.predict(df)

            return float(y_pred[0])

        predict_xgboost_udf = udf(predict_xgboost, FloatType())
        self.spark.udf.register("predict_xgboost_udf", predict_xgboost_udf)

        #spark_setup_delta = datetime.datetime.now() - now
        #print(f'spark setup time: {spark_setup_delta.total_seconds()}')



        if self.target_table == 'idnet':
            df = self.spark.read.parquet("../silver/labeled_training_set_no_tensor.parquet")

            self.labeled_training_set = df

        if self.target_table == 'idnet10k':
            df = self.spark.read.parquet("../silver/labeled_training_set_no_tensor_10k.parquet")

            self.labeled_training_set = df

        self.join_df = self.spark.read.parquet("../silver/alt_docs_x5.parquet")

        #self.labeled_training_set.cache()
        #self.labeled_training_set.count()

    def close(self):
        self.spark.stop()

    def query_1(self):
        """
        query 1: simple CNN inference in projection

        """
        self.setup()
        self.spark.sql("select gender, predict_udf(imageData) as my_predictions from {labeled_training_set}",
                  labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()


    def query_2(self):
        """
        query 2: CNN inference in CTE, followed by aggregation

        """
        self.setup()
        self.spark.sql("""with cte as (select gender, predict_udf(imageData) as my_predictions from {labeled_training_set})
                     select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions"""
                  , labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()

    # %%
    def query_3(self):
        """
        query 3: simple XGBOOST inference in projection

        """
        self.setup()
        self.spark.sql("""select gender, predict_xgboost_udf(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs)
            as my_predictions from {labeled_training_set}"""
                  , labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()

    # %%
    def query_4(self):
        """
        query 4: xgboost inference in CTE, followed by aggregation



        """
        self.setup()
        self.spark.sql("""with cte as (select gender,
                     predict_xgboost_udf(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) as my_predictions
                      from {labeled_training_set}) select gender, my_predictions, count(my_predictions) as count_predictions from cte
                     group by gender, my_predictions"""
                  , labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()

    # %%
    def query_5(self):
        """
        query 5: CNN inference in CTE, followed by aggregation, followed by structured predicate

        """
        self.setup()
        self.spark.sql("""with cte as (select gender, weight_lbs, predict_udf(imageData) as my_predictions from {labeled_training_set})
                     select gender, my_predictions, count(my_predictions) as count_predictions from cte where weight_lbs > 150
                     group by gender, my_predictions"""
                  , labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()

    def query_6(self):
        """

        query 6: Join to synthetic toll audit data, return all columns, find max of both CNN and XGBOOST with features intermixed
        between both tables in projection.

        """
        self.setup()
        result = self.spark.sql("""
        select lst.*, jd.*, greatest(
                predict_udf(lst.imageData)
            ,predict_xgboost_udf(lst.gender, lst.ethnicity, lst.class, jd.driver_eye_color, jd.driver_hair_color, lst.is_donor, lst.is_veteran, lst.height_inches, jd.driver_weight)
                            ) from {labeled_training_set} as lst
        inner join {join_df} as jd on lst.license_number = jd.license_number
        """, join_df=self.join_df, labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()

    def query_7(self):
        """
        query 7: Join to synthetic toll audit data, return all columns, CNN predict in projection

        """

        self.setup()
        result = self.spark.sql("""
        select lst.*, jd.*, predict_udf(lst.imageData) as my_predictions
            from {labeled_training_set} as lst
        inner join {join_df} as jd on lst.license_number = jd.license_number
        """, join_df=self.join_df, labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()

    def query_8(self):
        """

        query 8: Join to synthetic toll audit data, return all columns, XGBOOST predict in projection

        """
        self.setup()
        result = self.spark.sql("""
        select lst.*, jd.*, predict_xgboost_udf(lst.gender, lst.ethnicity, lst.class, jd.driver_eye_color, jd.driver_hair_color, lst.is_donor, lst.is_veteran, lst.height_inches, jd.driver_weight) as my_predictions
            from {labeled_training_set} as lst
        inner join {join_df} as jd on lst.license_number = jd.license_number
        """, join_df=self.join_df, labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()

    def query_9(self):
        """

        query 9: Join to synthetic toll audit data, return all columns, CNN predict in predicate

        """
        self.setup()
        result = self.spark.sql("""
        select lst.*, jd.*
            from {labeled_training_set} as lst
        inner join {join_df} as jd on lst.license_number = jd.license_number
            where int(predict_udf(lst.imageData))==0
        """, join_df=self.join_df, labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()

    def query_10(self):
        """

        query 10: Join to synthetic toll audit data, return all columns, XGBOOST predict in predicate

        """
        self.setup()
        result = self.spark.sql("""
        select lst.*, jd.*
            from {labeled_training_set} as lst
        inner join {join_df} as jd on lst.license_number = jd.license_number
            where int(predict_xgboost_udf(lst.gender, lst.ethnicity, lst.class, jd.driver_eye_color, jd.driver_hair_color, lst.is_donor, lst.is_veteran, lst.height_inches, jd.driver_weight))==0
        """, join_df=self.join_df, labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()

    def query_11(self):
        """

        query 11: Join to synthetic toll audit data, CTE and aggregate using structured data, CNN in HAVING clause as predicate

        """
        self.setup()
        result = self.spark.sql("""with cte as (select lst.imageData, lst.gender, lst.eye_color, jd.driver_hair_color from {labeled_training_set} as lst
        inner join {join_df} as jd on lst.license_number = jd.license_number)
                     select imageData, gender, eye_color, count(driver_hair_color) as count_hair from cte group by imageData, gender, eye_color
                       having int(predict_udf(imageData))==0 """
                  , join_df=self.join_df, labeled_training_set=self.labeled_training_set)
        print(result.count())
        self.close()

    def query_12(self):
        """
        query 12: Join to synthetic toll audit data, CTE and aggregate using structured data, XGBOOST in HAVING clause as predicate

        """
        self.setup()
        result = self.spark.sql("""with cte as (select lst.class, lst.gender, lst.ethnicity, lst.class, jd.driver_eye_color, jd.driver_hair_color, lst.is_donor, lst.is_veteran, lst.height_inches, jd.driver_weight
                                                from {labeled_training_set} as lst
                                       inner join {join_df} as jd
                                   on lst.license_number = jd.license_number)
        select *, count(class) as count_class
        from cte
        group by gender, ethnicity, class, driver_eye_color, driver_hair_color, is_donor, is_veteran, height_inches, driver_weight
        having int(predict_xgboost_udf(gender, ethnicity, class, driver_eye_color, driver_hair_color, is_donor, is_veteran, height_inches, driver_weight))==0 """
                                , join_df=self.join_df, labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()

    def query_13(self):
        """

        query 13: Join to synthetic toll audit data, select all, use CNN in order by


        """
        self.setup()
        result = self.spark.sql("""
                                select lst.*, jd.*
                                from {labeled_training_set} as lst
           inner join {join_df} as jd
                                on lst.license_number = jd.license_number
                                    order by predict_udf(lst.imageData) desc
                                """, join_df=self.join_df, labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()

    def query_14(self):
        """

        query 14: Join to synthetic toll audit data, select all, use XGBOOST in order by

        """
        self.setup()
        result = self.spark.sql("""
                                select lst.*,
                                       jd.*
                                from {labeled_training_set} as lst
           inner join {join_df} as jd
                                on lst.license_number = jd.license_number
                                    order by predict_xgboost_udf(lst.gender, lst.ethnicity, lst.class, jd.driver_eye_color,
                                                           jd.driver_hair_color, lst.is_donor, lst.is_veteran,
                                                           lst.height_inches, jd.driver_weight) desc
                                """, join_df=self.join_df, labeled_training_set=self.labeled_training_set).write.format("noop").mode("overwrite").save()
        self.close()


    #%%
    def run(self):
        global monitoring_active
        for i in range(1,15):
            self.log = []

            query = i
            print(f'{i} monitoring start')
            # Start the monitoring thread

            self.update_global(True)
            monitor_thread = threading.Thread(target=self.monitor_system_resources, args=(1,query))
            monitor_thread.start()

            # Run the function to be traced
            print(f'{i} start')
            if i == 1:

                self.query_1()
            if i == 2:

                self.query_2()
            if i == 3:

                self.query_3()
            if i == 4:

                self.query_4()
            if i == 5:

                self.query_5()
            if i == 6:
                self.query_6()
            if i == 7:
                self.query_7()
            if i == 8:
                self.query_8()
            if i == 9:
                self.query_9()
            if i == 10:
                self.query_10()
            if i == 11:
                self.query_11()
            if i == 12:
                self.query_12()
            if i == 13:
                self.query_13()
            if i == 14:
                self.query_14()
            print(f'{i} stop')

            # Stop the monitoring thread
            print('set false')
            self.update_global(False)

            monitor_thread.join() # Wait for the monitoring thread to finish
            print(f'{i} monitoring end')

            with open(f"/runlogs/{self.target_table}_{self.run_num}_{self.system}_{self.udf_type}_{query}.json", "w") as outfile:
                json.dump(self.log[:], outfile, indent=4)




In [9]:
SparkDB_Naive('idnet', 99).query_11()

799


In [6]:
SparkDB_Naive('idnet', 99).query_7()

25/12/16 14:37:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


6730


+--------------------+--------------------+----------------+--------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+--------------+------------+-------+-------------------+--------------------+-----+----------------+-----------------+----------------+-------------+---------------+--------------+
|                name|           imageData|prelabeled_fraud|                  id|    person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|license_number|plate_number|toll_id|          timestamp|            photo_id|speed|pass_device_type|driver_hair_color|driver_eye_color|driver_weight|toll_amount_usd|my_predictions|
+--------------------+--------------------+----------------+--------------------+---------------+--------------------+--------

In [7]:
SparkDB_Naive('idnet', 99).query_8()

25/12/16 14:39:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


6730


+--------------------+--------------------+----------------+--------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+--------------+------------+-------+-------------------+--------------------+-----+----------------+-----------------+----------------+-------------+---------------+--------------+
|                name|           imageData|prelabeled_fraud|                  id|    person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|license_number|plate_number|toll_id|          timestamp|            photo_id|speed|pass_device_type|driver_hair_color|driver_eye_color|driver_weight|toll_amount_usd|my_predictions|
+--------------------+--------------------+----------------+--------------------+---------------+--------------------+--------

In [8]:
SparkDB_Naive('idnet', 99).query_9()

25/12/16 14:39:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


4545


+--------------------+--------------------+----------------+--------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+--------------+------------+-------+-------------------+--------------------+-----+----------------+-----------------+----------------+-------------+---------------+
|                name|           imageData|prelabeled_fraud|                  id|    person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|license_number|plate_number|toll_id|          timestamp|            photo_id|speed|pass_device_type|driver_hair_color|driver_eye_color|driver_weight|toll_amount_usd|
+--------------------+--------------------+----------------+--------------------+---------------+--------------------+----------+------+---------+-----+----

In [9]:
SparkDB_Naive('idnet', 99).query_10()

25/12/16 14:40:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


6730


+--------------------+--------------------+----------------+--------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+--------------+------------+-------+-------------------+--------------------+-----+----------------+-----------------+----------------+-------------+---------------+
|                name|           imageData|prelabeled_fraud|                  id|    person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|license_number|plate_number|toll_id|          timestamp|            photo_id|speed|pass_device_type|driver_hair_color|driver_eye_color|driver_weight|toll_amount_usd|
+--------------------+--------------------+----------------+--------------------+---------------+--------------------+----------+------+---------+-----+----

In [10]:
SparkDB_Naive('idnet', 99).query_11()

25/12/16 14:48:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


799


+--------------------+------+---------+----------+
|           imageData|gender|eye_color|count_hair|
+--------------------+------+---------+----------+
|iVBORw0KGgoAAAANS...|     F|      BLU|         5|
|iVBORw0KGgoAAAANS...|     F|      BLU|         5|
|iVBORw0KGgoAAAANS...|     F|      BRO|        10|
|iVBORw0KGgoAAAANS...|     F|      BRO|         5|
|iVBORw0KGgoAAAANS...|     F|      BRO|         5|
|iVBORw0KGgoAAAANS...|     F|      BRO|         5|
|iVBORw0KGgoAAAANS...|     F|      BRO|         5|
|iVBORw0KGgoAAAANS...|     M|      BRO|         5|
|iVBORw0KGgoAAAANS...|     M|      BRO|         5|
|iVBORw0KGgoAAAANS...|     M|      BRO|         5|
|iVBORw0KGgoAAAANS...|     M|      BRO|         5|
|iVBORw0KGgoAAAANS...|     F|      BRO|         5|
|iVBORw0KGgoAAAANS...|     F|      BRO|         5|
|iVBORw0KGgoAAAANS...|     F|      BRO|         5|
|iVBORw0KGgoAAAANS...|     F|      GRA|         5|
|iVBORw0KGgoAAAANS...|     F|      BRO|         5|
|iVBORw0KGgoAAAANS...|     F|  

In [11]:
SparkDB_Naive('idnet', 99).query_12()

25/12/16 15:05:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


1027


+-----+------+---------+-----+----------------+-----------------+--------+----------+-------------+-------------+-----------+
|class|gender|ethnicity|class|driver_eye_color|driver_hair_color|is_donor|is_veteran|height_inches|driver_weight|count_class|
+-----+------+---------+-----+----------------+-----------------+--------+----------+-------------+-------------+-----------+
|    C|     F|   latino|    C|             GRY|              BLN|    TRUE|     FALSE|           64|          290|          5|
|    D|     F|    white|    D|             BLU|              BLK|   FALSE|     FALSE|           62|          169|          5|
|    D|     F|    white|    D|             BLU|              BLK|   FALSE|     FALSE|           91|          144|         30|
|    C|     M|    black|    C|             BLU|              BLK|   FALSE|     FALSE|           81|          222|          5|
|    B|     M|    white|    B|             GRY|              BLN|   FALSE|     FALSE|           66|          207|     

In [12]:
SparkDB_Naive('idnet', 99).query_13()

25/12/16 15:10:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


6730


+------------------+--------------------+----------------+------------------+-------------+--------------------+---------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+--------------+------------+-------+-------------------+--------------------+-----+----------------+-----------------+----------------+-------------+---------------+
|              name|           imageData|prelabeled_fraud|                id|  person_name|             address| birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|license_number|plate_number|toll_id|          timestamp|            photo_id|speed|pass_device_type|driver_hair_color|driver_eye_color|driver_weight|toll_amount_usd|
+------------------+--------------------+----------------+------------------+-------------+--------------------+---------+------+---------+-----+----------+-----------+--

In [13]:
SparkDB_Naive('idnet', 99).query_14()

25/12/16 15:20:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


6730


+--------------------+--------------------+----------------+--------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+--------------+------------+-------+-------------------+--------------------+-----+----------------+-----------------+----------------+-------------+---------------+
|                name|           imageData|prelabeled_fraud|                  id|    person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|license_number|plate_number|toll_id|          timestamp|            photo_id|speed|pass_device_type|driver_hair_color|driver_eye_color|driver_weight|toll_amount_usd|
+--------------------+--------------------+----------------+--------------------+---------------+--------------------+----------+------+---------+-----+----

In [11]:
#run spark.stop before running this notebook. some of the configs won't take if an active spark sessions is grabbed or running.
#spark.stop()
import psutil
import time
import threading
import datetime
import json
import xgboost as xgb
import matplotlib.pyplot as plt
import duckdb
import pandas as pd
import pandas as pd

import pyarrow as pa

import duckdb

import os

from torchgen.api.cpp import return_type
from torchvision.io import read_image
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import datasets, models
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid

import numpy as np
from PIL import Image
import pyarrow
from io import BytesIO
import base64

import xgboost as xgb
import pandas as pd

import pyarrow as pa
from duckdb.typing import *

import sys
sys.path.append('../')

from dbms_benchmark_parent_classes.dbms_parent import DMBSParent


def predict_tensor(base64image):
    class CNN(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.model = torch.nn.Sequential(
                # Input = 3 x 32 x 32, Output = 32 x 32 x 32
                # new input 3x600x375 output=
                torch.nn.Conv2d(in_channels=3, out_channels=6, kernel_size=3, padding='same'),
                torch.nn.ReLU(),
                # Input = 32 x 32 x 32, Output = 32 x 16 x 16
                torch.nn.MaxPool2d(kernel_size=2),

                # Input = 32 x 16 x 16, Output = 64 x 16 x 16
                torch.nn.Conv2d(in_channels=6, out_channels=12, kernel_size=3, padding='same'),
                torch.nn.ReLU(),
                # Input = 64 x 16 x 16, Output = 64 x 8 x 8
                torch.nn.MaxPool2d(kernel_size=2),

                # Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels=12, out_channels=24, kernel_size=3, padding='same'),
                torch.nn.ReLU(),
                # Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                # Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels=24, out_channels=48, kernel_size=3, padding='same'),
                torch.nn.ReLU(),
                # Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                # Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels=48, out_channels=96, kernel_size=3, padding='same'),
                torch.nn.ReLU(),
                # Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                torch.nn.Flatten(),

                torch.nn.Linear(96 * 11 * 18, 128),
                torch.nn.ReLU(),
                torch.nn.BatchNorm1d(128),

                # torch.nn.Linear(128, 128),
                # torch.nn.ReLU(),
                # torch.nn.BatchNorm1d(128),

                torch.nn.Linear(128, 2),
                torch.nn.ReLU(),
                # torch.nn.BatchNorm1d(2),
                # torch.nn.Dropout(0.2),

                torch.nn.Softmax(dim=1)

            )

        def forward(self, x):
            return self.model(x)

    img_data = base64.b64decode(base64image)
    image = Image.open(BytesIO(img_data))
    tensor = ToTensor()(image).float()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = CNN().to(device)
    model.load_state_dict(torch.load('/models/mymodel', map_location=torch.device(device)))
    model.eval()  # Set the model to evaluation mode

    with torch.no_grad():  # Disable gradient calculation for inference
        prediction = model(tensor.unsqueeze(0).to(device))
        _, y_pred = torch.max(prediction.data, 1)

    return float(y_pred)


def predict_xgboost(gender, ethnicity, driver_class, eye_color, hair_color, is_donor, is_veteran, height_inches,
                    weight_lbs):
    xgboostmodel = xgb.XGBClassifier(enable_categorical=True)
    xgboostmodel.load_model('/models/xgboost_model.model')

    data = [
        {
            'gender': gender,
            'ethnicity': ethnicity,
            'class': driver_class,
            'eye_color': eye_color,
            'hair_color': hair_color,
            'is_donor': is_donor,
            'is_veteran': is_veteran,
            'height_inches': height_inches,
            'weight_lbs': weight_lbs
        }
    ]
    df = pd.DataFrame(data)

    df['gender'] = df['gender'].astype('category')
    df['ethnicity'] = df['ethnicity'].astype('category')
    df['class'] = df['class'].astype('category')
    df['eye_color'] = df['eye_color'].astype('category')
    df['hair_color'] = df['hair_color'].astype('category')
    df['is_donor'] = df['is_donor'].astype('category')
    df['is_veteran'] = df['is_veteran'].astype('category')
    df['height_inches'] = df['height_inches'].astype('int64')
    df['weight_lbs'] = df['weight_lbs'].astype('int64')

    # feature_np = np.array(feature_list, dtype=object)

    y_pred = xgboostmodel.predict(df)
    # print(y_pred[0])

    return float(y_pred[0])


class DuckDB_Naive(DMBSParent):
    def __init__(self, target_table, run_num):
        super().__init__()
        self.target_table = target_table
        self.run_num = run_num
        self.system = "duckdb"
        self.udf_type = "naive"
        self.join_df = "toll_audit"

    def query_1(self):
        """
        query 1: simple CNN inference in projection

        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_torch_naive',
                function=predict_tensor,
                parameters=['STRING'],
                return_type='FLOAT'
            )
            result = qcon.sql(f'select predict_with_torch_naive(imagedata) from {self.target_table}')
            result.execute()

    def query_2(self):
        """
        query 2: CNN inference in CTE, followed by aggregation

        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_torch_naive',
                function=predict_tensor,
                parameters=['STRING'],
                return_type='FLOAT'
            )
            result = qcon.sql(
                f"""with cte as (select gender, predict_with_torch_naive(imagedata) as my_predictions from {self.target_table}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions""")
            result.execute()

    def query_3(self):
        """
        query 3: simple XGBOOST inference in projection

        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_xgboost_naive',
                function=predict_xgboost,
                parameters=['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'INTEGER',
                            'VARCHAR'],
                return_type='FLOAT'
            )
            result = qcon.sql(
                f'select predict_with_xgboost_naive(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) from {self.target_table}')
            result.execute()

    def query_4(self):
        """
        query 4: xgboost inference in CTE, followed by aggregation



        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_xgboost_naive',
                function=predict_xgboost,
                parameters=['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'INTEGER',
                            'VARCHAR'],
                return_type='FLOAT'
            )
            result = qcon.sql(
                f'with cte as (select gender, predict_with_xgboost_naive(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) as my_predictions from {self.target_table}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions')
            result.execute()

    def query_5(self):
        """
        query 5: CNN inference in CTE, followed by aggregation, followed by structured predicate

        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_torch_naive',
                function=predict_tensor,
                parameters=['STRING'],
                return_type='FLOAT'
            )

            result = qcon.sql(
                f"""with cte as (select gender, weight_lbs, predict_with_torch_naive(imagedata) as my_predictions from {self.target_table}) select gender, my_predictions, count(my_predictions) as count_predictions from cte where cast(weight_lbs AS INTEGER) > 150 group by gender, my_predictions;""")
            result.execute()

    def query_6(self):
        """

        query 6: Join to synthetic toll audit data, return all columns, find max of both CNN and XGBOOST with features intermixed
        between both tables in projection.

        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_torch_naive',
                function=predict_tensor,
                parameters=['STRING'],
                return_type='FLOAT'
            )
            qcon.create_function(
                name='predict_with_xgboost_naive',
                function=predict_xgboost,
                parameters=['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'INTEGER',
                            'VARCHAR'],
                return_type='FLOAT'
            )
            result = qcon.sql(f"""
        select lst.*, jd.*, greatest(
                predict_with_torch_naive(lst.imageData)
            ,predict_with_xgboost_naive(lst.gender, lst.ethnicity, lst.class, jd.driver_eye_color, jd.driver_hair_color, lst.is_donor, lst.is_veteran, lst.height_inches, jd.driver_weight)
                            ) from {self.target_table} as lst
        inner join {self.join_df} as jd on lst.license_number = jd.license_number
        """)
            result.execute()

    def query_7(self):
        """
        query 7: Join to synthetic toll audit data, return all columns, CNN predict in projection

        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_torch_naive',
                function=predict_tensor,
                parameters=['STRING'],
                return_type='FLOAT'
            )

            result = qcon.sql(f"""
        select lst.*, jd.*, predict_with_torch_naive(lst.imageData) as my_predictions
            from {self.target_table} as lst
        inner join {self.join_df} as jd on lst.license_number = jd.license_number
        """)
            result.execute()

    def query_8(self):
        """

        query 8: Join to synthetic toll audit data, return all columns, XGBOOST predict in projection

        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_xgboost_naive',
                function=predict_xgboost,
                parameters=['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'INTEGER',
                            'VARCHAR'],
                return_type='FLOAT'
            )
            result = qcon.sql(f"""
        select lst.*, jd.*, predict_with_xgboost_naive(lst.gender, lst.ethnicity, lst.class, jd.driver_eye_color, jd.driver_hair_color, lst.is_donor, lst.is_veteran, lst.height_inches, jd.driver_weight) as my_predictions
            from {self.target_table} as lst
        inner join {self.join_df} as jd on lst.license_number = jd.license_number
        """)
            result.execute()

    def query_9(self):
        """

        query 9: Join to synthetic toll audit data, return all columns, CNN predict in predicate

        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_torch_naive',
                function=predict_tensor,
                parameters=['STRING'],
                return_type='FLOAT'
            )
            result = qcon.sql(f"""
        select lst.*, jd.*
            from {self.target_table} as lst
        inner join {self.join_df} as jd on lst.license_number = jd.license_number
            where cast(predict_with_torch_naive(lst.imageData) as integer)==0
        """)
            result.execute()

    def query_10(self):
        """

        query 10: Join to synthetic toll audit data, return all columns, XGBOOST predict in predicate

        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_xgboost_naive',
                function=predict_xgboost,
                parameters=['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'INTEGER',
                            'VARCHAR'],
                return_type='FLOAT'
            )
            result = qcon.sql(f"""
        select lst.*, jd.*
            from {self.target_table} as lst
        inner join {self.join_df} as jd on lst.license_number = jd.license_number
            where cast(predict_with_xgboost_naive(lst.gender, lst.ethnicity, lst.class, jd.driver_eye_color, jd.driver_hair_color, lst.is_donor, lst.is_veteran, lst.height_inches, jd.driver_weight) as integer)==0
        """)
            result.execute()

    def query_11(self):
        """

        query 11: Join to synthetic toll audit data, CTE and aggregate using structured data, CNN in HAVING clause as predicate

        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_torch_naive',
                function=predict_tensor,
                parameters=['STRING'],
                return_type='FLOAT'
            )
            result = qcon.sql(
                f"""with cte as (select lst.imageData, lst.gender, lst.eye_color, jd.driver_hair_color from {self.target_table} as lst
        inner join {self.join_df} as jd on lst.license_number = jd.license_number)
                     select imageData, gender, eye_color, count(driver_hair_color) as count_hair from cte group by imageData, gender, eye_color
                       having cast(predict_with_torch_naive(imageData) as integer)==0 """)
            result.execute()

    def query_12(self):
        """
        query 12: Join to synthetic toll audit data, CTE and aggregate using structured data, XGBOOST in HAVING clause as predicate

        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_xgboost_naive',
                function=predict_xgboost,
                parameters=['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'INTEGER',
                            'VARCHAR'],
                return_type='FLOAT'
            )
            result = qcon.sql(f"""with cte as (select lst.class, lst.gender, lst.ethnicity, jd.driver_eye_color, jd.driver_hair_color, lst.is_donor, lst.is_veteran, lst.height_inches, jd.driver_weight
                                                from {self.target_table} as lst
                                       inner join {self.join_df} as jd
                                   on lst.license_number = jd.license_number)
        select *, count(class) as count_class
        from cte
        group by gender, ethnicity, class, driver_eye_color, driver_hair_color, is_donor, is_veteran, height_inches, driver_weight
        having cast(predict_with_xgboost_naive(gender, ethnicity, class, driver_eye_color, driver_hair_color, is_donor, is_veteran, height_inches, driver_weight) as integer)==0 """)
            result.execute()

    def query_13(self):
        """

        query 13: Join to synthetic toll audit data, select all, use CNN in order by


        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_torch_naive',
                function=predict_tensor,
                parameters=['STRING'],
                return_type='FLOAT'
            )

            result = qcon.sql(f"""
                                select lst.*, jd.*
                                from {self.target_table} as lst
           inner join {self.join_df} as jd
                                on lst.license_number = jd.license_number
                                    order by predict_with_torch_naive(lst.imageData) desc
                                """)
            result.execute()

    def query_14(self):
        """

        query 14: Join to synthetic toll audit data, select all, use XGBOOST in order by

        """
        with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_xgboost_naive',
                function=predict_xgboost,
                parameters=['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'INTEGER',
                            'VARCHAR'],
                return_type='FLOAT'
            )
            result = qcon.sql(f"""
                                select lst.*,
                                       jd.*
                                from {self.target_table} as lst
           inner join {self.join_df} as jd
                                on lst.license_number = jd.license_number
                                    order by predict_with_xgboost_naive(lst.gender, lst.ethnicity, lst.class, jd.driver_eye_color,
                                                           jd.driver_hair_color, lst.is_donor, lst.is_veteran,
                                                           lst.height_inches, jd.driver_weight) desc
                                """)
            result.execute()

    #%%
    def run(self):
        for i in range(6, 15):
            self.log = []

            query = i
            print(f'{i} monitoring start')
            self.update_global(True)
            monitor_thread = threading.Thread(target=self.monitor_system_resources, args=(1, query))
            monitor_thread.start()

            # Run the function to be traced
            print(f'{i} start')
            if i == 1:
                self.query_1()
            if i == 2:
                self.query_2()
            if i == 3:
                self.query_3()
            if i == 4:
                self.query_4()
            if i == 5:
                self.query_5()
            if i == 6:
                self.query_6()
            if i == 7:
                self.query_7()
            if i == 8:
                self.query_8()
            if i == 9:
                self.query_9()
            if i == 10:
                self.query_10()
            if i == 11:
                self.query_11()
            if i == 12:
                self.query_12()
            if i == 13:
                self.query_13()
            if i == 14:
                self.query_14()
            print(f'{i} stop')

            # Stop the monitoring thread
            self.update_global(False)
            monitor_thread.join()  # Wait for the monitoring thread to finish
            print(f'{i} monitoring end')

            with open(f"/runlogs/{self.target_table}_{self.run_num}_{self.system}_{self.udf_type}_{query}.json",
                      "w") as outfile:
                json.dump(self.log[:], outfile, indent=4)


/tmp/ipykernel_3645/3446779667.py:39: DeprecationWarning: `duckdb.typing` is deprecated and will be removed in a future version. Please use `duckdb.sqltypes` instead.
  from duckdb.typing import *


In [12]:
DuckDB_Naive('idnet', 99).query_5()

In [13]:
with duckdb.connect("../duckdatabase/idnet.db") as qcon:

            result = qcon.sql(f"""show idnet""")
            result.show()

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ name             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ imageData        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ prelabeled_fraud │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ id               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ person_name      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ address          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ birthday         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ gender           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ethnicity        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ class            │ VARC

In [19]:
with duckdb.connect("../duckdatabase/idnet.db") as qcon:
            qcon.create_function(
                name='predict_with_torch_naive',
                function=predict_tensor,
                parameters=['STRING'],
                return_type='FLOAT'
            )

            qcon.execute("PRAGMA enable_profiling = 'json';")
            qcon.execute("PRAGMA profiling_output='query_plan.json' ;")
            result = qcon.sql(
                f"""with cte as (select gender, height_inches, predict_with_torch_naive(imagedata) as my_predictions from idnet) select gender, my_predictions, count(my_predictions) as count_predictions from cte where height_inches > 72 group by gender, my_predictions;""")
            result.execute()


In [2]:
from postgres_naive import PostgresDB_Naive

In [3]:
PostgresDB_Naive('idnet', 99).query_1()

In [4]:
PostgresDB_Naive('idnet', 99).query_3()

In [1]:
from evadb_naive import EvaDB_Naive
from evadb_optimized import EvaDB_Optimized

In [4]:
EvaDB_Optimized('idnet', 99).query_1()

/home/research-temp/PycharmProjects/mdpbenchmark/.venv/lib/python3.10/site-packages/evadb/third_party/databases/postgres/postgres_handler.py:93: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables_df = pd.read_sql_query(query, self.connection)
/home/research-temp/PycharmProjects/mdpbenchmark/.venv/lib/python3.10/site-packages/evadb/third_party/databases/postgres/postgres_handler.py:111: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  columns_df = pd.read_sql_query(query, self.connection)


In [5]:
EvaDB_Optimized('idnet', 99).query_3()

/home/research-temp/PycharmProjects/mdpbenchmark/.venv/lib/python3.10/site-packages/evadb/third_party/databases/postgres/postgres_handler.py:93: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables_df = pd.read_sql_query(query, self.connection)
/home/research-temp/PycharmProjects/mdpbenchmark/.venv/lib/python3.10/site-packages/evadb/third_party/databases/postgres/postgres_handler.py:111: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  columns_df = pd.read_sql_query(query, self.connection)


In [6]:
from duckdb_naive import DuckDB_Naive
from duckdb_optimized import DuckDB_Optimized

In [7]:
DuckDB_Optimized('idnet', 99).query_1()

In [8]:
DuckDB_Optimized('idnet', 99).query_3()

In [9]:
from clickhouse_naive import ClickHouseDB_Naive

In [10]:
ClickHouseDB_Naive('idnet', 99).query_1()

In [11]:
ClickHouseDB_Naive('idnet', 99).query_3()

In [16]:
import clickhouse_connect

from clickhouse_connect.driver.tools import insert_file
import pyarrow
import pandas as pd

import matplotlib.pyplot as plt

import psutil
import time
import threading
import datetime
import json

from dbms_benchmark_parent_classes.dbms_parent import DMBSParent


class ClickHouseDB_Naive(DMBSParent):
    def __init__(self, target_table, run_num):
        super().__init__()
        self.target_table = target_table
        self.run_num = run_num
        self.system = "clickhouse"
        self.udf_type = "naive"
        self.join_df = "toll_audit"




    def query_1(self):
        client = clickhouse_connect.get_client(host='localhost', username='default', password='', send_receive_timeout=14400)
        result = client.query(f'select gender, torch_function_python(name) as my_predictions from {self.target_table}').result_rows
        df = pd.DataFrame(result)
        df
        client.close()

    def query_2(self):
        client = clickhouse_connect.get_client(host='localhost', username='default', password='', send_receive_timeout=14400)
        result = client.query(
            f'with cte as (select gender, torch_function_python(name) as my_predictions from {self.target_table}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions').result_rows
        df = pd.DataFrame(result)
        df
        client.close()

    # %%
    def query_3(self):
        client = clickhouse_connect.get_client(host='localhost', username='default', password='', send_receive_timeout=14400)
        result = client.query(
            f"select gender, xgboost_function_python(gender, ifNull(ethnicity, 'None') as ethnicity, class, eye_color, ifNull(hair_color,'None') as hair_color, is_donor, is_veteran, height_inches, weight_lbs) from {self.target_table}").result_rows
        df = pd.DataFrame(result)
        df
        client.close()

    # %%
    def query_4(self):
        client = clickhouse_connect.get_client(host='localhost', username='default', password='', send_receive_timeout=14400)

        result = client.query(
            f"with cte as (select gender, xgboost_function_python(gender, ifNull(ethnicity, 'None') as ethnicity, class, eye_color, ifNull(hair_color,'None') as hair_color, is_donor, is_veteran, height_inches, weight_lbs) as my_predictions from {self.target_table}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions").result_rows
        df = pd.DataFrame(result)
        df
        client.close()

    # %%
    def query_5(self):
        client = clickhouse_connect.get_client(host='localhost', username='default', password='', send_receive_timeout=14400)

        result = client.query(
            f'with cte as (select gender, weight_lbs, torch_function_python(name) as my_predictions from {self.target_table}) select gender, my_predictions, count(my_predictions) as count_predictions from cte where (weight_lbs > 150) group by gender, my_predictions').result_rows
        df = pd.DataFrame(result)
        df
        client.close()

    def query_6(self):
        """

        query 6: Join to synthetic toll audit data, return all columns, find max of both CNN and XGBOOST with features intermixed
        between both tables in projection.

        """
        client = clickhouse_connect.get_client(host='localhost', username='default', password='',
                                               send_receive_timeout=14400)

        result = client.query(
            f"""
        select lst.*, jd.*, greatest(
                torch_function_python(lst.name)
            ,xgboost_function_python(lst.gender, ifNull(lst.ethnicity,'None') as ethnicity, lst.class, jd.driver_eye_color, ifNull(jd.driver_hair_color,'None') as driver_hair_color, lst.is_donor, lst.is_veteran, lst.height_inches, jd.driver_weight)
                            ) from {self.target_table} as lst
        inner join {self.join_df} as jd on lst.license_number = jd.license_number
        """).result_rows
        df = pd.DataFrame(result)
        df
        client.close()




    def query_7(self):
        """
        query 7: Join to synthetic toll audit data, return all columns, CNN predict in projection

        """

        client = clickhouse_connect.get_client(host='localhost', username='default', password='',
                                               send_receive_timeout=14400)

        result = client.query(
            f"""
        select lst.*, jd.*, torch_function_python(lst.name) as my_predictions
            from {self.target_table} as lst
        inner join {self.join_df} as jd on lst.license_number = jd.license_number
        """).result_rows
        df = pd.DataFrame(result)
        df
        client.close()





    def query_8(self):
        """

        query 8: Join to synthetic toll audit data, return all columns, XGBOOST predict in projection

        """
        client = clickhouse_connect.get_client(host='localhost', username='default', password='',
                                               send_receive_timeout=14400)

        result = client.query(
            f"""
        select lst.*, jd.*, xgboost_function_python(lst.gender, ifNull(lst.ethnicity, 'None') as ethnicity, lst.class, jd.driver_eye_color, ifNull(driver_hair_color,'None') as driver_hair_color, lst.is_donor, lst.is_veteran, lst.height_inches, jd.driver_weight) as my_predictions
            from {self.target_table} as lst
        inner join {self.join_df} as jd on lst.license_number = jd.license_number
        """).result_rows
        df = pd.DataFrame(result)
        df
        client.close()



    def query_9(self):
        """

        query 9: Join to synthetic toll audit data, return all columns, CNN predict in predicate

        """

        client = clickhouse_connect.get_client(host='localhost', username='default', password='',
                                               send_receive_timeout=14400)

        result = client.query(
            f"""
        select lst.*, jd.*
            from {self.target_table} as lst
        inner join {self.join_df} as jd on lst.license_number = jd.license_number
            where torch_function_python(lst.name)::numeric::integer = 0
        """).result_rows
        df = pd.DataFrame(result)
        df
        client.close()




    def query_10(self):
        """

        query 10: Join to synthetic toll audit data, return all columns, XGBOOST predict in predicate

        """
        client = clickhouse_connect.get_client(host='localhost', username='default', password='',
                                               send_receive_timeout=14400)

        result = client.query(
            f"""
        select lst.*, jd.*
            from {self.target_table} as lst
        inner join {self.join_df} as jd on lst.license_number = jd.license_number
            where xgboost_function_python(lst.gender, ifNull(lst.ethnicity, 'None') as ethnicity, lst.class, jd.driver_eye_color, ifNull(driver_hair_color,'None') as driver_hair_color, lst.is_donor, lst.is_veteran, lst.height_inches, jd.driver_weight)::numeric::integer = 0
        """).result_rows
        df = pd.DataFrame(result)
        df
        client.close()




    def query_11(self):
        """

        query 11: Join to synthetic toll audit data, CTE and aggregate using structured data, CNN in HAVING clause as predicate

        """

        client = clickhouse_connect.get_client(host='localhost', username='default', password='',
                                               send_receive_timeout=14400)

        result = client.query(
            f"""with cte as (select lst.name, lst.imageData, lst.gender, lst.eye_color, jd.driver_hair_color from {self.target_table} as lst
        inner join {self.join_df} as jd on lst.license_number = jd.license_number)
                     select name, gender, eye_color, count(driver_hair_color) as count_hair from cte group by name, gender, eye_color
                       having torch_function_python(name)::numeric::integer = 0 """).result_rows
        df = pd.DataFrame(result)
        df
        print(len(df))
        client.close()



    def query_12(self):
        """
        query 12: Join to synthetic toll audit data, CTE and aggregate using structured data, XGBOOST in HAVING clause as predicate

        """
        client = clickhouse_connect.get_client(host='localhost', username='default', password='',
                                               send_receive_timeout=14400)

        result = client.query(
            f"""with cte as (select lst.class, lst.gender, lst.ethnicity, jd.driver_eye_color, jd.driver_hair_color, lst.is_donor, lst.is_veteran, lst.height_inches, jd.driver_weight
                                                from {self.target_table} as lst
                                       inner join {self.join_df} as jd
                                   on lst.license_number = jd.license_number)
        select gender, ethnicity, class, driver_eye_color, driver_hair_color, is_donor, is_veteran, height_inches, driver_weight, count(class) as count_class
        from cte
        group by gender, ethnicity, class, driver_eye_color, driver_hair_color, is_donor, is_veteran, height_inches, driver_weight
        having xgboost_function_python(gender, ifNull(ethnicity, 'None') as ethnicity, class, driver_eye_color, ifNull(driver_hair_color,'None') as driver_hair_color, is_donor, is_veteran, height_inches, driver_weight)::numeric::integer = 0 """).result_rows
        df = pd.DataFrame(result)
        df
        client.close()




    def query_13(self):
        """

        query 13: Join to synthetic toll audit data, select all, use CNN in order by


        """
        client = clickhouse_connect.get_client(host='localhost', username='default', password='',
                                               send_receive_timeout=14400)

        result = client.query(
            f"""
                                select lst.*, jd.*
                                from {self.target_table} as lst
           inner join {self.join_df} as jd
                                on lst.license_number = jd.license_number
                                    order by torch_function_python(lst.name) desc
                                """).result_rows
        df = pd.DataFrame(result)
        df
        client.close()





    def query_14(self):
        """

        query 14: Join to synthetic toll audit data, select all, use XGBOOST in order by

        """
        client = clickhouse_connect.get_client(host='localhost', username='default', password='',
                                               send_receive_timeout=14400)

        result = client.query(
            f"""
                                select lst.*,
                                       jd.*
                                from {self.target_table} as lst
           inner join {self.join_df} as jd
                                on lst.license_number = jd.license_number
                                    order by xgboost_function_python(lst.gender, ifNull(lst.ethnicity, 'None') as ethnicity, lst.class, jd.driver_eye_color,
                                                           ifNull(driver_hair_color,'None') as driver_hair_color, lst.is_donor, lst.is_veteran,
                                                           lst.height_inches, jd.driver_weight) desc
                                """).result_rows
        df = pd.DataFrame(result)
        df
        client.close()





    #%%
    def run(self):
        for i in range(1,15):
        #for i in [1]:
            self.log = []

            query = i
            print(f'{i} monitoring start')
            self.update_global(True)
            monitor_thread = threading.Thread(target=self.monitor_system_resources, args=(1,query))
            monitor_thread.start()

            # Run the function to be traced
            print(f'{i} start')
            if i == 1:
                self.query_1()
            if i == 2:
                self.query_2()
            if i == 3:
                self.query_3()
            if i == 4:
                self.query_4()
            if i == 5:
                self.query_5()
            if i == 6:
                self.query_6()
            if i == 7:
                self.query_7()
            if i == 8:
                self.query_8()
            if i == 9:
                self.query_9()
            if i == 10:
                self.query_10()
            if i == 11:
                self.query_11()
            if i == 12:
                self.query_12()
            if i == 13:
                self.query_13()
            if i == 14:
                self.query_14()
            print(f'{i} stop')

            # Stop the monitoring thread
            self.update_global(False)
            monitor_thread.join() # Wait for the monitoring thread to finish
            print(f'{i} monitoring end')

            with open(f"/runlogs/{self.target_table}_{self.run_num}_{self.system}_{self.udf_type}_{query}.json", "w") as outfile:
                json.dump(self.log[:], outfile, indent=4)



In [17]:
ClickHouseDB_Naive('idnet', 99).query_11()

799
